# 【notebook｜N-22】（PyTorch 从零）
> 对应：【定义 1.1.3｜D-1.1.3】多层感知机、【代码 1.1.4｜Cd-1.1.4】`MLP.forward`、【定义 1.2.4｜D-1.2.4】交叉熵与负对数似然、【定义 1.2.5｜D-1.2.5】均方误差、【定理 1.3.4｜T-1.3.4】反向传播递推；代码 `F-pre-py5`

> **对应文档**：[`PY5-PyTorch从零.md`](../../00-预备/PY5-PyTorch从零.md)
> **对应代码**：[`py5_torch_from_zero.py`](../../00-预备/code/py5_torch_from_zero.py)
> **Colab**：CPU 运行时即可，全部运行约 1 分钟（无 GPU 自动走 CPU 路径）。

## 学习目标
1. 分清 `Tensor` 的 shape / dtype / device，以及与 NumPy 的共享与拷贝语义；
2. 理解 `view/reshape/transpose/contiguous` 与广播，能解释「为什么 view 会失败」；
3. 掌握 autograd：`requires_grad` / `grad_fn` / `backward` / 梯度累加 / `detach` / `no_grad`；
4. 用 `nn.Module` 组织模型，走通「前向 - 损失 - 清零 - 反向 - 更新」五步；
5. 手写线性回归与两层 MLP 两个训练闭环，并保存/加载 checkpoint。


## 1. 环境检查与 Tensor 基础

`Tensor` 是「同构缓冲区 + shape/stride 元数据 + dtype/device/requires_grad」；详见【定义 PY5.1.1】。

In [ ]:
# 【N-22.c1】

import sys
print("Python", sys.version.split()[0])
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
except ModuleNotFoundError:
    print("未安装 torch，请先运行：pip install torch --index-url https://download.pytorch.org/whl/cpu")
    raise


In [ ]:
# 【N-22.c2】

import numpy as np
import torch

t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("shape/dtype/device:", t.shape, t.dtype, t.device)
print("默认浮点 dtype:", torch.get_default_dtype())          # float32（NumPy 默认 float64）

arr = np.ones((2, 3), dtype=np.float32)
t_np = torch.from_numpy(arr)                                  # 零拷贝，共享内存
arr[0, 0] = 9.0
print("from_numpy 共享内存:", t_np[0, 0].item() == 9.0)
print("clone 独立:", t.clone().data_ptr() != t.data_ptr())
print("detach 共享数据:", t.detach().data_ptr() == t.data_ptr())


## 2. 形状操作与广播

`view` 只在连续内存上可用；`transpose` 只改 stride 会得到非连续张量；规则与 NumPy 一致（见【注 PY5.2.5】）。

In [ ]:
# 【N-22.c3】

import torch

x = torch.arange(12).reshape(3, 4)
print("view(4,3):", x.view(4, 3).shape)
print("transpose stride:", x.t().stride())
try:
    x.t().view(-1)                                            # 非连续，view 会失败
except RuntimeError as e:
    print("view 非连续失败:", str(e)[:70])
print("contiguous().view:", x.t().contiguous().view(-1).shape)
print("广播 (3,4)+(4,):", (x + torch.ones(4)).shape)
print("unsqueeze:", x.unsqueeze(0).shape, "-> squeeze:", x.unsqueeze(0).squeeze(0).shape)


## 3. autograd：计算图与梯度

`backward()` 在【定理 1.3.4】的计算图上做反向模式链式法则；梯度是**累加**的，训练时每步前须清零（【注 PY5.3.4】）。

In [ ]:
# 【N-22.c4】

import torch

x = torch.tensor([2.0], requires_grad=True)
y = (x * x + 3 * x).sum()
print("grad_fn:", y.grad_fn)
y.backward()
print("dy/dx =", x.grad.item())                # 2x+3 = 7

x.grad = None
(x * x).sum().backward()
(x * x).sum().backward()
print("不清零 -> 梯度累加:", x.grad.item())     # 4 + 4 = 8

with torch.no_grad():
    print("no_grad 下不记录:", (x * 2).requires_grad)
print("数值梯度对照:", ((x.detach() + 1e-4) ** 2 - (x.detach() - 1e-4) ** 2).item() / 2e-4)


## 4. nn.Module：参数注册与 state_dict

把 `nn.Parameter` 或子模块赋为属性即自动注册；`state_dict` 是「名字 → 张量」。与【定义 1.1.3】的 MLP 同构。

In [ ]:
# 【N-22.c5】

import torch
import torch.nn as nn

class TwoLayerMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, out_dim))
    def forward(self, x):
        return self.net(x)

model = TwoLayerMLP(4, 16, 3)
print("参数名:", [n for n, _ in model.named_parameters()])
print("参数量:", sum(p.numel() for p in model.parameters()))
print("state_dict keys:", list(model.state_dict().keys()))
model.train(); model.eval()
print("train/eval 切换成功")


## 5. 数据管线：Dataset 与 DataLoader

`__len__` + `__getitem__` 即 Dataset 协议；`DataLoader` 负责 batch / shuffle / collate（【定义 PY5.6.2】）。

In [ ]:
# 【N-22.c6】

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

ds = TensorDataset(torch.randn(100, 4), torch.randint(0, 3, (100,)))
dl = DataLoader(ds, batch_size=16, shuffle=True, num_workers=0)
xb, yb = next(iter(dl))
print("内置 TensorDataset batch:", xb.shape, yb.shape)

class ToyDataset(Dataset):
    def __init__(self):
        self.x = torch.randn(50, 4)
        self.y = (self.x[:, 0] > 0).long()
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]

print("自定义 Dataset 样本数:", len(ToyDataset()))


## 6. 训练闭环一：线性回归

标准五步：前向 → 损失 → 清零 → 反向 → 更新（【算法 PY5.7.1】）。

In [ ]:
# 【N-22.c7】

import torch
import torch.nn as nn

torch.manual_seed(0)
x = torch.rand(200, 1) * 10 - 5
y = 3 * x + 2 + 0.1 * torch.randn(200, 1)

model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

for step in range(1, 301):
    pred = model(x)
    loss = loss_fn(pred, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss={loss.item():.5f}")

print("weight=%.3f (真值 3)  bias=%.3f (真值 2)" % (model.weight.item(), model.bias.item()))


## 7. 训练闭环二：两层 MLP 分类

`CrossEntropyLoss` 吃 logits（【定义 PY5.5.1】）；验证用 `eval()` + `no_grad()`；`state_dict` 用于保存/加载。

In [ ]:
# 【N-22.c8】

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)

class Clusters(Dataset):
    def __init__(self, n=120):
        centers = torch.tensor([[2.0, 0.0], [-1.0, 1.8], [-1.0, -1.8]])
        xs = [centers[c] + 0.6 * torch.randn(n, 2) for c in range(3)]
        ys = [torch.full((n,), c, dtype=torch.long) for c in range(3)]
        self.x, self.y = torch.cat(xs), torch.cat(ys)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]

train_ds, val_ds = torch.utils.data.random_split(Clusters(), [288, 72])
dl_tr = DataLoader(train_ds, batch_size=64, shuffle=True)
dl_va = DataLoader(val_ds, batch_size=64)

model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 3))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate():
    model.eval()
    correct = n = 0
    for xb, yb in dl_va:
        correct += (model(xb).argmax(1) == yb).sum().item()
        n += xb.size(0)
    model.train()
    return correct / n

for epoch in range(1, 9):
    model.train()
    for xb, yb in dl_tr:
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    print(f"epoch {epoch}  val_acc={evaluate():.3f}")

torch.save(model.state_dict(), "py5_mlp.pt")
model2 = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 3))
model2.load_state_dict(torch.load("py5_mlp.pt", map_location="cpu"))
print("checkpoint 加载成功，state_dict 键数:", len(model2.state_dict()))


## 8. 常见错误诊断

对照【注 PY5.8.1】~【注 PY5.8.5】。运行下面单元，观察报错文本。

In [ ]:
# 【N-22.c9】

import torch

x = torch.tensor([1.0], requires_grad=True)
try:
    x += 1.0                                     # 叶子原地修改
except RuntimeError as e:
    print("in-place on leaf:", str(e)[:70])

try:
    torch.nn.Linear(4, 3)(torch.randn(2, 5))     # in_features=4 与输入最后一维 5 不符
except RuntimeError as e:
    print("size mismatch:", str(e)[:70])

import torch.nn as nn
m = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 2))
try:
    m(torch.randn(3, 2, dtype=torch.float64))    # dtype 不一致
except RuntimeError as e:
    print("dtype mismatch:", str(e)[:70])


## 9. 小结与练习

1. 五步骨架 `forward → loss → zero_grad → backward → step` 是所有训练循环的核心；
2. 手写并对照 `F-pre-py5` 的两个闭环；
3. 练习：【练习 PY5.10.1】~【练习 PY5.10.6】。